In [1]:
!pip install -U -q git+https://github.com/huggingface/transformers

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
!pip install -q accelerate qwen-vl-utils bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 41.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 76.1 MB/s eta 0:00:00:00:0100:01


In [ ]:
"""
Qwen3-VL 캡션 생성 (LookMatch) — 로컬 / Colab / 구글드라이브 겸용
================================================================
crop된 옷 이미지 + 네이버 상품명(힌트) → 깨끗한 한국어 캡션 생성.
네이버 제목의 SEO 키워드 나열을 CLIP 학습에 좋은 캡션으로 정제.

특징:
  - 실행 환경 자동 감지 (로컬 / Colab / 구글드라이브)
    · 로컬: 현재 폴더의 clothes_dataset/ 사용
    · Colab: 구글드라이브 자동 연결 → MyDrive/lookmatch/ 사용
    · 환경변수 LOOKMATCH_DIR 로 경로 직접 지정도 가능
  - VRAM에 따라 Qwen3-VL 모델 자동 선택 (8B / 4B / 2B)
  - 배치 처리로 속도 ↑
  - 이어받기 (중간에 끊겨도 재실행하면 이어서)
  - 이미지 경로 자동 보정

모델 (VRAM 기준 자동):
  20GB+ → Qwen3-VL-8B-Instruct  (bf16, 품질 우선)
  12GB+ → Qwen3-VL-8B + 4bit    (8B 4bit ≈ 6GB)
  8GB+  → Qwen3-VL-4B-Instruct  (균형)
  그 미만 → Qwen3-VL-2B-Instruct (최경량)

입력:  <데이터폴더>/metadata_cropped.csv (crop_path, caption)
출력:  <데이터폴더>/metadata_captioned.csv (+ qwen_caption 컬럼)

사용법:
  [로컬]
    pip install -U git+https://github.com/huggingface/transformers
    pip install accelerate qwen-vl-utils bitsandbytes pillow pandas tqdm torch
    python generate_captions.py

  [Colab]
    !pip install -U -q git+https://github.com/huggingface/transformers
    !pip install -q accelerate qwen-vl-utils bitsandbytes
    !python generate_captions.py
    # → 드라이브 자동 연결, MyDrive/lookmatch/ 사용

  [경로 직접 지정]
    LOOKMATCH_DIR=/my/path python generate_captions.py

※ Qwen3-VL은 최신 transformers 필요 (구버전이면 모델 로드 실패).
"""

import os
import gc
import json
from pathlib import Path

import pandas as pd
import torch
from PIL import Image
from tqdm import tqdm
import re

# ============================================================
# 경로 설정 — 로컬 / Colab / 구글드라이브 자동 감지
# ============================================================
# 우선순위:
#   1) 환경변수 LOOKMATCH_DIR 이 있으면 그걸 사용
#   2) Colab이고 드라이브에 lookmatch 폴더가 있으면 드라이브 사용
#   3) Colab이면 드라이브 연결 후 드라이브 사용
#   4) 로컬이면 현재 폴더의 clothes_dataset 사용
def setup_data_dir():
    # 1) 환경변수 우선 (사용자가 직접 지정 가능)
    env_dir = os.environ.get("LOOKMATCH_DIR")
    if env_dir:
        print(f"데이터 경로(환경변수): {env_dir}")
        return Path(env_dir)

    # Colab 여부 확인
    try:
        import google.colab  # noqa
        is_colab = True
    except ImportError:
        is_colab = False

    if is_colab:
        # 2~3) Colab → 드라이브 연결
        drive_root = Path("/content/drive/MyDrive")
        if not drive_root.exists():
            print("구글 드라이브 연결 중... (인증 창에서 허용해주세요)")
            from google.colab import drive
            drive.mount("/content/drive")
        # 드라이브 안 lookmatch 폴더 (없으면 생성)
        data_dir = drive_root / "clothes_dataset"
        data_dir.mkdir(parents=True, exist_ok=True)
        print(f"데이터 경로(구글드라이브): {data_dir}")
        return data_dir

    # 4) 로컬
    data_dir = Path("clothes_dataset")
    print(f"데이터 경로(로컬): {data_dir.resolve()}")
    return data_dir


DATA_DIR = setup_data_dir()
IN_CSV = str(DATA_DIR / "metadata_cropped.csv")
OUT_CSV = str(DATA_DIR / "metadata_captioned.csv")
PROGRESS_FILE = str(DATA_DIR / "caption_progress.json")

# crop 이미지 경로가 다른 환경에서 만들어졌을 때 보정용
# (예: 로컬에서 크롭 → 드라이브에서 캡션 → crop_path가 로컬 경로로 남아있음)
# CSV의 crop_path가 절대경로로 안 맞으면, 파일명만 떼어 DATA_DIR/crops 에서 찾음
CROPS_SUBDIR = "crops"

BATCH_SIZE = 4           # 배치 크기 (OOM 나면 2로 더 줄이기)
MAX_NEW_TOKENS = 120     # 5개 항목 분석 출력 (조합은 후처리가 하므로 과하지 않게)
SAVE_EVERY = 200         # N개마다 중간 저장 (이어받기용)
LIMIT = 100                # 테스트용: 0이면 전체, 100이면 앞 100개만 처리


def resolve_image_path(crop_path):
    """crop_path가 현재 환경에서 안 맞을 때 파일명으로 재탐색.
    윈도우(\\)·리눅스(/) 경로 혼용도 처리."""
    crop_path = str(crop_path)
    # 1) 있는 그대로 존재하면 사용
    if Path(crop_path).exists():
        return str(Path(crop_path))
    # 2) 윈도우 백슬래시(\)와 리눅스 슬래시(/) 모두에서 파일명만 추출
    #    (예: "clothes_dataset\crops\abc.jpg" → "abc.jpg")
    fname = crop_path.replace("\\", "/").split("/")[-1]
    # 3) DATA_DIR/crops 에서 파일명으로 찾기
    candidate = DATA_DIR / CROPS_SUBDIR / fname
    if candidate.exists():
        return str(candidate)
    return None  # 못 찾음

# 프롬프트 — 멀티모달 구조화 분석 (조합은 후처리가 담당)
# 모델은 각 속성만 정확히 채우게 하고, 캡션 조합은 코드(clean_caption)가 수행.
# → 모델이 최종 조합을 빠뜨리는 문제 방지 + 항목별 정확도 향상
# + 같은 옷이 여러 색으로 나열된 이미지는 '나열'로 표시 → 후처리에서 제외
# PROMPT_TEMPLATE = (
#     "이 옷 이미지를 보고 아래 항목을 한국어로 채워줘.\n"
#     "먼저, 같은 옷이 여러 색상으로 나란히 나열된 이미지인지 판단해줘.\n"
#     "그런 '색상 옵션 나열' 이미지면 아래처럼만 답해:\n"
#     "나열=예\n\n"
#     "하나의 옷만 있으면 아래 5개 항목을 채워줘.\n"
#     "각 항목은 짧은 단어로만. 안 보이거나 해당 없으면 '없음'.\n"
#     "'캐주얼·데일리·편안한' 같은 추상어와 브랜드명은 쓰지 마.\n\n"
#     "나열=아니오\n"
#     "1.색상= (주요 색상 1~2개)\n"
#     "2.종류= (슬랙스/니트/코트 등 옷 종류, 상품명 참고)\n"
#     "3.핏= (와이드/오버핏/슬림 등, 없으면 없음)\n"
#     "4.패턴= (무지/체크/스트라이프/데님 등)\n"
#     "5.디테일= (밴딩/하이웨이스트/라운드넥 등 보이는 것, 없으면 없음)\n\n"
#     "참고 상품명: {title}\n\n"
#     "위 형식 그대로 '1.색상= ...' 부터 '5.디테일= ...' 까지만 답해:"
# )
# 프롬프트 — 영어 지시 + 한국어 출력 (판단 정확도 향상)
# 영어로 명확히 지시하되, 캡션 값은 한국어로 출력하게 함.
# 색상 옵션 나열 이미지는 MULTI=yes로 표시 → 후처리에서 제외.
# PROMPT_TEMPLATE = (
#     "Look at this clothing product image carefully.\n\n"
#     "STEP 1: Decide if this image shows the SAME clothing item repeated in "
#     "MULTIPLE COLORS side by side (a color-option lineup, e.g. the same pants "
#     "in black, yellow, beige, white lined up).\n"
#     "- If YES (multiple color variants of the same item): answer ONLY:\n"
#     "MULTI=yes\n"
#     "- If NO (a single item, or one person wearing clothes): continue to STEP 2.\n\n"
#     "STEP 2: Fill in these fields IN KOREAN with short words only. "
#     "Use '없음' if not visible. Do NOT use abstract words "
#     "(캐주얼/데일리/편안한) or brand names.\n"
#     "MULTI=no\n"
#     "1.색상= (main color in Korean)\n"
#     "2.종류= (garment type in Korean: 슬랙스/니트/코트 etc, refer to product name)\n"
#     "3.핏= (fit: 와이드/오버핏/슬림, or 없음)\n"
#     "4.패턴= (pattern: 무지/체크/스트라이프/데님)\n"
#     "5.디테일= (details: 밴딩/하이웨이스트/라운드넥, or 없음)\n\n"
#     "Product name (reference): {title}\n\n"
#     "Answer in the exact format above:"
# )
PROMPT_TEMPLATE = (
    "Look at this clothing product image carefully.\n\n"
    "STEP 1: Decide if this image shows the SAME clothing item repeated in "
    "MULTIPLE COLORS side by side (a color-option lineup, e.g. the same pants "
    "in black, yellow, beige, white lined up).\n"
    "- If YES (multiple color variants of the same item): answer ONLY:\n"
    "MULTI=yes\n"
    "- If NO (a single item, or one person wearing clothes): continue to STEP 2.\n\n"
    "STEP 2: Fill in these fields IN KOREAN with short words only. "
    "Use '없음' if not visible. Do NOT use abstract words "
    "(캐주얼/데일리/편안한) or brand names.\n"
    "The examples below are just samples, NOT a fixed list — "
    "choose the most accurate Korean word for THIS item, even if it is not listed.\n"
    "Use the product name only as a hint; always prioritize what you actually SEE in the image.\n"
    "MULTI=no\n"
    "1.색상= (main color in Korean, e.g. 베이지/네이비/카키 etc)\n"
    "2.종류= (garment type in Korean, e.g. 슬랙스/니트/코트 etc)\n"
    "3.핏= (fit in Korean, e.g. 와이드/세미와이드/오버핏/슬림/스트레이트/테이퍼드/스키니/부츠컷/세미부츠컷 etc, or 없음)\n"
    "4.패턴= (pattern in Korean, e.g. 무지/체크/스트라이프/데님/꽃무늬 etc)\n"
    "5.디테일= (details in Korean, e.g. 밴딩/하이웨이스트/라운드넥/카라/포켓 etc, or 없음)\n\n"
    "Product name (hint only, image comes first): {title}\n\n"
    "Answer in the exact format above:"
)


# ============================================================
# 환경 감지
# ============================================================
def detect_env():
    """Colab인지, GPU 있는지, VRAM 얼마인지 감지"""
    try:
        import google.colab  # noqa
        is_colab = True
    except ImportError:
        is_colab = False

    has_gpu = torch.cuda.is_available()
    vram_gb = 0
    if has_gpu:
        vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9

    print(f"환경: {'Colab' if is_colab else '로컬'}")
    print(f"GPU: {'있음' if has_gpu else '없음 (CPU - 매우 느림)'}")
    if has_gpu:
        print(f"VRAM: {vram_gb:.1f}GB ({torch.cuda.get_device_name(0)})")
    return is_colab, has_gpu, vram_gb


# ============================================================
# 모델 선택 & 로드 (VRAM에 맞게)
# ============================================================
def load_model(vram_gb):
    """VRAM에 따라 Qwen3-VL 모델 크기·양자화 자동 선택"""
    from transformers import AutoModelForImageTextToText, AutoProcessor

    # VRAM 기준 Qwen3-VL 모델 선택
    # (bf16 기준 대략: 8B~18GB지만 이미지 처리 오버헤드로 실제 24GB도 빠듯)
    # → 8B bf16은 28GB+ 에서만, 그 아래는 4bit으로 안전하게
    if vram_gb >= 28:
        model_id = "Qwen/Qwen3-VL-8B-Instruct"
        quant = None
        print(f"→ Qwen3-VL-8B (VRAM 충분, 품질 우선)")
    elif vram_gb >= 10:
        model_id = "Qwen/Qwen3-VL-8B-Instruct"
        quant = "4bit"
        print(f"→ Qwen3-VL-8B + 4bit 양자화 (8B 4bit ≈ 6GB, 안전)")
    elif vram_gb >= 8:
        model_id = "Qwen/Qwen3-VL-4B-Instruct"
        quant = None
        print(f"→ Qwen3-VL-4B (균형)")
    else:
        model_id = "Qwen/Qwen3-VL-2B-Instruct"
        quant = None
        print(f"→ Qwen3-VL-2B (VRAM 제한, 최경량)")

    kwargs = {"device_map": "auto", "torch_dtype": torch.bfloat16}

    # 4bit 양자화 (VRAM 절약)
    if quant == "4bit":
        from transformers import BitsAndBytesConfig
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
        )

    # 이미지 토큰 수 제한 (속도·메모리)
    min_pixels = 256 * 28 * 28
    max_pixels = 768 * 28 * 28

    model = AutoModelForImageTextToText.from_pretrained(model_id, **kwargs)
    processor = AutoProcessor.from_pretrained(
        model_id, min_pixels=min_pixels, max_pixels=max_pixels
    )
    # 배치 생성(decoder)에서는 left padding이 필수
    # (right padding이면 PAD 토큰 위치에서 생성이 시작돼 출력이 깨짐)
    if hasattr(processor, "tokenizer"):
        processor.tokenizer.padding_side = "left"
    model.eval()
    return model, processor


# ============================================================
# 캡션 정제 (후처리)
# ============================================================
def clean_caption(text):
    """구조화 분석 출력(1.색상=.. 2.종류=..)에서 값을 추출해
    [색상] [패턴] [핏] [디테일] [종류] 순서로 캡션 조합.
    '나열=예'(색상 옵션 나열 이미지)면 빈 문자열 반환 → 제외."""
    text = text.strip()

    # 색상 옵션 나열 이미지 감지 → 제외 (빈 캡션)
    if re.search(r"나열\s*[=:]\s*예", text) or "나열=예" in text.replace(" ", ""):
        return ""

    def grab(label):
        # "색상=베이지", "색상 : 베이지" 등 다양한 형태 대응
        m = re.search(rf"{label}\s*[=:]\s*([^\n0-9]+?)(?:\s*\d+\.|$|\n)", text)
        if not m:
            return ""
        v = m.group(1).strip(" .,·:=[]\"'")
        # '없음'류 제거
        if "없" in v:
            return ""
        # 추상어 제거
        for a in ["캐주얼", "데일리", "베이직", "심플", "편안한", "여름", "겨울", "봄", "가을"]:
            v = v.replace(a, "")
        return v.strip()

    color   = grab("색상")
    gtype   = grab("종류")
    fit     = grab("핏")
    pattern = grab("패턴")
    detail  = grab("디테일")

    # [색상] [패턴] [핏] [디테일] [종류] — 종류는 마지막
    parts = [color, pattern, fit, detail, gtype]
    caption = " ".join(p for p in parts if p)

    # 최종 정제
    for junk in ['"', "'", "`", "*", "•", "·", ":", "[", "]", "=", ","]:
        caption = caption.replace(junk, " ")
    caption = " ".join(caption.split())
    return caption.strip(" .·:")[:80]


# ============================================================
# 배치 캡션 생성
# ============================================================
def caption_batch(model, processor, rows):
    """이미지+제목 배치 → 캡션 리스트"""
    messages_list = []
    valid_idx = []
    for i, row in enumerate(rows):
        img_path = resolve_image_path(row["crop_path"])
        if img_path is None:
            continue
        title = str(row.get("caption", ""))[:100]  # 네이버 제목(힌트)
        messages_list.append([{
            "role": "user",
            "content": [
                {"type": "image", "image": img_path},
                {"type": "text", "text": PROMPT_TEMPLATE.format(title=title)},
            ],
        }])
        valid_idx.append(i)

    if not messages_list:
        return {}

    # 배치 전처리
    from qwen_vl_utils import process_vision_info
    texts = [processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
             for m in messages_list]
    image_inputs, video_inputs = process_vision_info(messages_list)
    inputs = processor(
        text=texts, images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt",
    ).to(model.device)

    # 생성
    with torch.no_grad():
        gen_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, gen_ids)]
    outputs = processor.batch_decode(trimmed, skip_special_tokens=True,
                                     clean_up_tokenization_spaces=False)

    # idx → 캡션 매핑
    result = {}
    for local_i, cap in zip(valid_idx, outputs):
        result[local_i] = clean_caption(cap)
    return result


# ============================================================
# 메인
# ============================================================
def main():
    is_colab, has_gpu, vram_gb = detect_env()
    if not has_gpu:
        print("\n⚠️ GPU가 없어 매우 느립니다. Colab GPU 런타임을 권장합니다.")

    # 테스트 모드면 출력/진행 파일을 별도로 (전체 실행과 안 섞이게)
    out_csv = OUT_CSV
    progress_file = PROGRESS_FILE
    if LIMIT > 0:
        out_csv = OUT_CSV.replace(".csv", "_test.csv")
        progress_file = PROGRESS_FILE.replace(".json", "_test.json")

    df = pd.read_csv(IN_CSV)
    if LIMIT > 0:
        df = df.head(LIMIT).copy()
        print(f"[테스트 모드] 앞 {LIMIT}개만 처리 → {out_csv}")
    print(f"\n총 {len(df)}개 캡션 생성 대상\n")

    # 이어받기: 이미 처리한 인덱스 로드
    done = set()
    captions = {}
    if Path(progress_file).exists():
        with open(progress_file, encoding="utf-8") as f:
            saved = json.load(f)
            captions = {int(k): v for k, v in saved.items()}
            done = set(captions.keys())
        print(f"이어받기: {len(done)}개 이미 완료\n")

    # 모델 로드
    model, processor = load_model(vram_gb)

    # 배치 처리
    todo = [i for i in range(len(df)) if i not in done]
    for start in tqdm(range(0, len(todo), BATCH_SIZE), desc="캡션 생성"):
        batch_idx = todo[start:start + BATCH_SIZE]
        rows = [df.iloc[i].to_dict() for i in batch_idx]

        try:
            batch_result = caption_batch(model, processor, rows)
            for local_i, cap in batch_result.items():
                captions[batch_idx[local_i]] = cap
        except torch.cuda.OutOfMemoryError:
            print(f"\nOOM! BATCH_SIZE를 줄이세요 (현재 {BATCH_SIZE}). 진행분 저장.")
            torch.cuda.empty_cache()
            break
        except Exception as e:
            print(f"\n배치 오류(건너뜀): {e}")
            continue

        # 중간 저장 (이어받기용)
        if (start // BATCH_SIZE) % (SAVE_EVERY // BATCH_SIZE + 1) == 0:
            with open(progress_file, "w", encoding="utf-8") as f:
                json.dump({str(k): v for k, v in captions.items()}, f, ensure_ascii=False)
            gc.collect()
            if has_gpu:
                torch.cuda.empty_cache()

    # 최종 저장
    with open(progress_file, "w", encoding="utf-8") as f:
        json.dump({str(k): v for k, v in captions.items()}, f, ensure_ascii=False)

    df["qwen_caption"] = df.index.map(lambda i: captions.get(i, ""))
    df.to_csv(out_csv, index=False, encoding="utf-8-sig")

    # 요약 + 샘플
    n_done = sum(1 for v in captions.values() if v)
    print(f"\n{'='*55}")
    print(f"캡션 생성 완료: {n_done}/{len(df)}개")
    print(f"저장: {out_csv}")
    print(f"\n[샘플 5개 — 네이버 제목 → Qwen 캡션]")
    shown = 0
    for i in range(len(df)):
        if captions.get(i):
            orig = str(df.iloc[i].get("caption", ""))[:40]
            print(f"  원본: {orig}")
            print(f"  정제: {captions[i]}\n")
            shown += 1
            if shown >= 5:
                break
    print("="*55)


if __name__ == "__main__":
    main()

데이터 경로(구글드라이브): /content/drive/MyDrive/clothes_dataset
환경: Colab
GPU: 있음
VRAM: 23.7GB (NVIDIA L4)
[테스트 모드] 앞 100개만 처리 → /content/drive/MyDrive/clothes_dataset/metadata_captioned_test.csv

총 100개 캡션 생성 대상

→ Qwen3-VL-8B + 4bit 양자화 (8B 4bit ≈ 6GB, 안전)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

캡션 생성: 100%|██████████| 25/25 [03:31<00:00,  8.48s/it]


캡션 생성 완료: 89/100개
저장: /content/drive/MyDrive/clothes_dataset/metadata_captioned_test.csv

[샘플 5개 — 네이버 제목 → Qwen 캡션]
  원본: 개미다리로 보이는 여자 린넨 반 바지 바지 밴딩 여름 바지 빅사이즈 슬렉
  정제: 베이지 무지 와이드 밴딩/포켓 반바지

  원본: 냉장고 일자 세미 와이드 밴딩팬츠
  정제: 블랙 무지 와이드 밴딩 팬츠

  원본: 폴햄 쿨링 린렌라이크 밴딩팬츠 PHG2PP2089
  정제: 그레이 무지 와이드 밴딩 팬츠

  원본: 탑텐 리넨라이크 와이드 밴딩 팬츠 MSF2PP2602
  정제: 검정 무지 와이드 밴딩 팬츠

  원본: 플라워 플리츠 꽃무늬 팬츠 여름 냉장고 주름 바지 밴딩 고무줄 촌캉스 원
  정제: 빨강 꽃무늬 와이드 밴딩 팬츠

